# Chain Rule & Automatic Differentiation

The canonical implementation is `julia code/main.jl`; the Python helper mirrors its scalar `Value` operations.

In [ ]:
from pathlib import Path
import sys
candidates = (Path.cwd() / 'code', Path.cwd().parents[1] / 'code', Path.cwd() / 'phases/01-math-foundations/05-chain-rule-and-autodiff/code')
code_dir = next(path for path in candidates if (path / 'autodiff.py').is_file())
sys.path.insert(0, str(code_dir))
from autodiff import Value


## Build a graph

Each operator creates a node with a local backward closure. Start with the same fixture as the Julia demo.

In [ ]:
x1, x2 = Value(2.0), Value(3.0)
y = (x1 * x2 + 1).relu()
y.backward()
y.data, x1.grad, x2.grad


The result is `(7.0, 3.0, 2.0)`: the upstream gradient is multiplied by the opposite input at the product node.

In [ ]:
x = Value(2.0)
power = x ** 3
power.backward()
power.data, x.grad


## Gradient checking

Compare a reverse-mode result with a centered finite difference before extending the engine. Gradients must be reset before a second backward pass on shared parameters.

In [ ]:
def finite_difference(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)
finite_difference(lambda z: z ** 3, 2.0)


## Exercise

Change the ReLU preactivation to a negative value and verify that the local implementation sends zero gradient through ReLU. Then add two consumers of one `Value` and check that its gradient accumulates.